In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2'
outdir = '/u/project/cluo/terencew/claude/igvf/pgp/ipsc_le_se_rerun/csv'
donors = ['C29', 'C37', 'C38', 'C39']
hicdir = f'{projdir}/snm3C/hicluster'

In [ ]:
nuclei = pd.read_csv(f'{outdir}/ipsc_le_se_all_nuclei.csv', sep='\t', index_col=0)
early = nuclei[nuclei['early_ipsc']]
nuclei.shape, early.shape

In [ ]:
early_3c = early[early['assay'] == 'snm3C']
early_3c.groupby(['line', 'ips_subtype']).size().unstack()

### per-nucleus contact distance distributions (as in 04_sort_ips_subtypes_contacts_by_short_long)

In [ ]:
log2_bins = np.arange(np.log2(2500), np.log2(250000000) + 0.125, 0.125)
bin_edges = 2 ** log2_bins
short_bins = bin_edges[bin_edges < 2e6][1:]
long_bins = bin_edges[bin_edges > 2e6]
contact_df = pd.read_csv(f'{hicdir}/csv/distance/merged.csv.gz', sep='\t', header=0, index_col=0)
contact_df.columns = bin_edges[1:]
contact_df.shape

In [ ]:
sum_df = pd.DataFrame({'short': contact_df.loc[:, short_bins].sum(axis=1), 'long': contact_df.loc[:, long_bins].sum(axis=1)})
sum_df['short/long'] = sum_df['short'] / sum_df['long']
ips_sum = sum_df.reindex(early_3c.index).dropna()
ips_sum['ips_subtype'] = early_3c['ips_subtype'].reindex(ips_sum.index)
ips_sum['line'] = early_3c['line'].reindex(ips_sum.index)
ips_sum.shape

In [ ]:
ips_sum.groupby('ips_subtype')['short/long'].describe()

In [ ]:
rows = []
for d in ['all'] + donors:
    x = ips_sum if d == 'all' else ips_sum[ips_sum['line'] == d]
    a = x.loc[x['ips_subtype'] == 'IPS_main', 'short/long']
    b = x.loc[x['ips_subtype'] == 'IPS_alt', 'short/long']
    rows.append([d, len(a), len(b), a.median(), b.median(), mannwhitneyu(a, b).pvalue])
ratio_stats = pd.DataFrame(rows, columns=['donor', 'n_LE', 'n_SE', 'median_short_long_LE', 'median_short_long_SE', 'p'])
ratio_stats

### z-scored distance profiles ordered by short/long within subtype (Fig 6E)

In [ ]:
cell_order = pd.concat([ips_sum[ips_sum['ips_subtype'] == s].sort_values('short/long') for s in ['IPS_main', 'IPS_alt']]).index
ips_df = contact_df.reindex(cell_order)
ips_zscore = ips_df.sub(ips_df.mean(axis=1), axis=0).div(ips_df.std(axis=1), axis=0).T
ips_zscore.shape

In [ ]:
ips_zscore.to_csv(f'{outdir}/ips_subtypes_dist_df.csv', sep='\t')
ips_sum.reindex(cell_order).to_csv(f'{outdir}/ips_subtypes_dist_meta.csv', sep='\t')
ratio_stats.to_csv(f'{outdir}/ips_subtypes_short_long_stats.csv', sep='\t', index=False)

In [ ]:
!date